<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_49_INDEPENDENT_VERIFICATION_AND_CROSS_EXAMINATION_READINESS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# EXPERIMENT 11
# THE EXPERT WITNESS:
# INDEPENDENT VERIFICATION AND CROSS-EXAMINATION READINESS
# ================================================================

import hashlib
import os
import re
import tempfile


# ================================================================
# PART A
# INDEPENDENT REPRODUCTION OF INTEGRITY RESULT
# ================================================================

def independent_verify(
    evidence_path,
    claimed_hash,
    claimed_algo="sha256"
):
    """
    Independently recompute the hash of an exhibit
    and compare it with the examiner's claimed hash.
    """

    h = hashlib.new(claimed_algo)

    with open(evidence_path, "rb") as f:

        for block in iter(
            lambda: f.read(1 << 20),
            b""
        ):
            h.update(block)

    actual = h.hexdigest()

    reproduced = (
        actual.lower()
        == claimed_hash.lower()
    )

    if reproduced:

        conclusion = (
            "The exhibit is bit-for-bit identical "
            "to the exhibit described in the report."
        )

    else:

        conclusion = (
            "The exhibit does NOT match the hash "
            "recorded in the report; the difference "
            "must be explained."
        )

    return {
        "algorithm": claimed_algo,
        "claimed": claimed_hash.lower(),
        "recomputed": actual,
        "reproduced": reproduced,
        "conclusion": conclusion
    }


# ================================================================
# PART B
# OPINION REVIEW
# ================================================================

OVERCLAIM = [

    r"\bproves?\b",
    r"\bconclusively\b",
    r"\bwithout doubt\b",
    r"\bcertainly\b",
    r"\bthe accused\b",
    r"\bguilty\b",
    r"\bobviously\b",
    r"\b100%\b",
    r"\bhacker\b"
]


GOOD_PRACTICE = [

    r"\bin my opinion\b",
    r"\bconsistent with\b",
    r"\bI express no opinion\b",
    r"\bbased on the (?:material|data|artefacts) examined\b",
    r"\blimitation",
    r"\bcannot be determined\b"
]


def review_opinion(text):

    t = text.lower()

    overclaims = [
        pattern
        for pattern in OVERCLAIM
        if re.search(pattern, t)
    ]

    good = [
        pattern
        for pattern in GOOD_PRACTICE
        if re.search(pattern, t)
    ]

    score = max(
        0,
        min(
            10,
            5 + 2 * len(good) - 3 * len(overclaims)
        )
    )

    if overclaims:

        risk = (
            "HIGH - statement exceeds what the "
            "technical evidence can support"
        )

    elif len(good) >= 2:

        risk = (
            "LOW - appropriately qualified and "
            "within the examiner's expertise"
        )

    else:

        risk = (
            "MEDIUM - add qualifications, basis "
            "and limitations"
        )

    return {
        "score_out_of_10": score,
        "risk": risk,
        "overclaim_markers": [
            p.strip("\\b")
            for p in overclaims
        ],
        "good_practice_markers": [
            p.strip("\\b")
            for p in good
        ]
    }


# ================================================================
# CROSS-EXAMINATION DRILL
# ================================================================

CROSS_EXAM_DRILL = [

    (
        "What are your qualifications to give this opinion?",
        "State qualifications, training and relevant "
        "casework only."
    ),

    (
        "Did you examine the original device or a copy?",
        "A verified working copy; the original was "
        "write-blocked."
    ),

    (
        "How do you know the copy is identical?",
        "SHA-256 computed at acquisition and "
        "re-verified before analysis."
    ),

    (
        "Could another person have used this account?",
        "That cannot be determined from the "
        "artefacts examined."
    ),

    (
        "Is your tool validated?",
        "State the tool, version and the validation "
        "or dual-tool check performed."
    ),

    (
        "What did you NOT examine, and why?",
        "State scope limits and unavailable data honestly."
    )
]


# ================================================================
# TEST CASES
# ================================================================

def run_tests():

    # ------------------------------------------------------------
    # CREATE TEMPORARY EVIDENCE
    # ------------------------------------------------------------

    temp_directory = tempfile.mkdtemp()

    evidence_path = os.path.join(
        temp_directory,
        "EX01.dd"
    )

    with open(evidence_path, "wb") as f:

        f.write(
            b"exhibit contents for reproduction test"
            * 100
        )

    # ------------------------------------------------------------
    # ORIGINAL HASH
    # ------------------------------------------------------------

    with open(evidence_path, "rb") as f:

        true_hash = hashlib.sha256(
            f.read()
        ).hexdigest()

    results = []

    # ============================================================
    # A - INTEGRITY REPRODUCTION
    # ============================================================

    result_ok = independent_verify(
        evidence_path,
        true_hash
    )

    results.append(
        (
            "TC1 matching hash is reproduced",
            result_ok["reproduced"] is True
        )
    )

    results.append(
        (
            "TC2 conclusion wording for a match",
            "bit-for-bit identical"
            in result_ok["conclusion"]
        )
    )

    # ------------------------------------------------------------
    # WRONG HASH
    # ------------------------------------------------------------

    result_bad = independent_verify(
        evidence_path,
        "0" * 64
    )

    results.append(
        (
            "TC3 mismatched hash is flagged",
            result_bad["reproduced"] is False
        )
    )

    results.append(
        (
            "TC4 mismatch demands explanation",
            "must be explained"
            in result_bad["conclusion"]
        )
    )

    # ============================================================
    # TAMPER WITH EXHIBIT
    # ============================================================

    with open(evidence_path, "r+b") as f:

        f.seek(5)
        f.write(b"X")

    tampered_result = independent_verify(
        evidence_path,
        true_hash
    )

    results.append(
        (
            "TC5 tampered exhibit fails reproduction",
            tampered_result["reproduced"] is False
        )
    )

    # ============================================================
    # B - OPINION REVIEW
    # ============================================================

    bad_opinion = (
        "This conclusively proves that the accused "
        "is the hacker who certainly stole the data. "
        "It is obviously his doing."
    )

    good_opinion = (
        "In my opinion the artefacts examined are "
        "consistent with an unauthorised transfer "
        "of data from the internal network. "
        "I express no opinion on the identity of "
        "the operator; that cannot be determined "
        "from the material examined. "
        "This opinion is subject to the limitations "
        "stated in section 10."
    )

    bad_review = review_opinion(
        bad_opinion
    )

    good_review = review_opinion(
        good_opinion
    )

    # ------------------------------------------------------------
    # PRINT OPINION RESULTS
    # ------------------------------------------------------------

    print("\n" + "=" * 78)
    print("OPINION REVIEW")
    print("=" * 78)

    print("\nOVERCLAIMING OPINION:")
    print(bad_opinion)

    print("\nRESULT:")
    print(bad_review)

    print("\nWELL-FRAMED OPINION:")
    print(good_opinion)

    print("\nRESULT:")
    print(good_review)

    # ------------------------------------------------------------
    # TC6
    # ------------------------------------------------------------

    results.append(
        (
            "TC6 overclaiming opinion rated HIGH risk",
            bad_review["risk"].startswith("HIGH")
        )
    )

    # ------------------------------------------------------------
    # TC7
    # ------------------------------------------------------------

    results.append(
        (
            "TC7 qualified opinion rated LOW risk",
            good_review["risk"].startswith("LOW")
        )
    )

    # ------------------------------------------------------------
    # TC8
    # ------------------------------------------------------------

    results.append(
        (
            "TC8 qualified opinion scores higher",
            good_review["score_out_of_10"]
            > bad_review["score_out_of_10"]
        )
    )

    # ------------------------------------------------------------
    # TC9
    # ------------------------------------------------------------

    results.append(
        (
            "TC9 overclaim markers identified",
            len(
                bad_review["overclaim_markers"]
            ) >= 3
        )
    )

    # ------------------------------------------------------------
    # TC10
    # ------------------------------------------------------------

    results.append(
        (
            "TC10 good-practice markers identified",
            len(
                good_review["good_practice_markers"]
            ) >= 2
        )
    )

    # ============================================================
    # TC11 - MEDIUM RISK
    # ============================================================

    middle_opinion = (
        "The file was uploaded at "
        "04:18 IST on 20-08-2026."
    )

    middle_review = review_opinion(
        middle_opinion
    )

    results.append(
        (
            "TC11 bare factual statement rated MEDIUM",
            middle_review["risk"].startswith("MEDIUM")
        )
    )

    # ============================================================
    # TC12 - CROSS EXAMINATION
    # ============================================================

    results.append(
        (
            "TC12 cross-examination drill complete",
            len(CROSS_EXAM_DRILL) >= 6
        )
    )

    # ============================================================
    # DISPLAY CROSS EXAMINATION DRILL
    # ============================================================

    print("\n" + "=" * 78)
    print("CROSS-EXAMINATION DRILL")
    print("=" * 78)

    for question, answer in CROSS_EXAM_DRILL:

        print("\nQ:", question)
        print("A:", answer)

    # ============================================================
    # DISPLAY TEST RESULTS
    # ============================================================

    print("\n" + "=" * 78)
    print("TEST CASE RESULTS")
    print("=" * 78)

    for name, passed in results:

        status = "PASS" if passed else "FAIL"

        print(
            f"{name:<50} -> {status}"
        )

    passed_count = sum(
        1
        for _, ok in results
        if ok
    )

    print("=" * 78)

    print(
        f"RESULT: {passed_count}/{len(results)} "
        "test cases passed"
    )

    return passed_count == len(results)


# ================================================================
# RUN EXPERIMENT
# ================================================================

print("=" * 78)
print("EXPERIMENT 11")
print("THE EXPERT WITNESS")
print("INDEPENDENT VERIFICATION AND CROSS-EXAMINATION READINESS")
print("=" * 78)

success = run_tests()

print("\n" + "=" * 78)

if success:

    print(
        "EXPERIMENT 11 COMPLETED SUCCESSFULLY"
    )

else:

    print(
        "EXPERIMENT 11 COMPLETED WITH FAILURES"
    )

print("=" * 78)

EXPERIMENT 11
THE EXPERT WITNESS
INDEPENDENT VERIFICATION AND CROSS-EXAMINATION READINESS

OPINION REVIEW

OVERCLAIMING OPINION:
This conclusively proves that the accused is the hacker who certainly stole the data. It is obviously his doing.

RESULT:
{'score_out_of_10': 0, 'risk': 'HIGH - statement exceeds what the technical evidence can support', 'overclaim_markers': ['proves?', 'conclusively', 'certainly', 'the accused', 'obviously', 'hacker'], 'good_practice_markers': []}

WELL-FRAMED OPINION:
In my opinion the artefacts examined are consistent with an unauthorised transfer of data from the internal network. I express no opinion on the identity of the operator; that cannot be determined from the material examined. This opinion is subject to the limitations stated in section 10.

RESULT:
{'score_out_of_10': 10, 'risk': "LOW - appropriately qualified and within the examiner's expertise", 'overclaim_markers': [], 'good_practice_markers': ['in my opinion', 'consistent with', 'limitation